In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    confusion_matrix,
    fbeta_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
N_SPLITS = 5
SPLIT_SEEDS = [19, 42, 117]
TARGET = "failure"
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 30)
print("Random state:", RANDOM_STATE)

Random state: 42


In [ ]:
DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "train" / "train.csv"
TEST_PATH = DATA_DIR / "test" / "test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

expected_features = [
    "core_temp",
    "hyd_pressure",
    "voltage_draw",
    "sync_rate",
    "coolant_level",
    "vibration",
    "operating_hours",
]

train = train[expected_features + [TARGET]].copy()
test = test[expected_features].copy()

print("Train shape:", train.shape)
print("Test shape: ", test.shape)
train.head()

Train shape: (14070, 8)
Test shape:  (6030, 7)


In [3]:
quality_report = pd.DataFrame({
    "dtype": train[expected_features].dtypes.astype(str),
    "train_missing": train[expected_features].isna().sum(),
    "train_missing_%": (100 * train[expected_features].isna().mean()).round(2),
    "test_missing": test[expected_features].isna().sum(),
    "test_missing_%": (100 * test[expected_features].isna().mean()).round(2),
    "train_unique": train[expected_features].nunique(),
})

class_report = pd.DataFrame({
    "count": train[TARGET].value_counts().sort_index(),
    "percent": (100 * train[TARGET].value_counts(normalize=True).sort_index()).round(2),
})

print("Duplicate full rows in train:", train.duplicated().sum())
print("Duplicate rows in test:      ", test.duplicated().sum())
print("\nClass distribution (0=healthy, 1=failure):")
display(class_report)
print("Missing values and data types:")
display(quality_report)

Duplicate full rows in train: 68
Duplicate rows in test:       2

Class distribution (0=healthy, 1=failure):
         count  percent
failure                
0        12036    85.54
1         2034    14.46
Missing values and data types:
                   dtype  train_missing  train_missing_%  test_missing  \
core_temp        float64            437             3.11           188   
hyd_pressure     float64              0             0.00             0   
voltage_draw     float64              0             0.00             0   
sync_rate        float64            394             2.80           188   
coolant_level    float64            439             3.12           178   
vibration        float64            382             2.71           197   
operating_hours  float64              0             0.00             0   

                 test_missing_%  train_unique  
core_temp                  3.12          5689  
hyd_pressure               0.00          8994  
voltage_draw               

In [ ]:
distribution_report = pd.DataFrame({
    "train_mean": train[expected_features].mean(),
    "test_mean": test[expected_features].mean(),
    "train_std": train[expected_features].std(),
    "test_std": test[expected_features].std(),
})
distribution_report["mean_shift_in_train_sd"] = (
    (distribution_report["test_mean"] - distribution_report["train_mean"])
    / distribution_report["train_std"].replace(0, np.nan)
)

print("Train/test distribution comparison:")
display(distribution_report.round(3))
print("Feature medians by target class:")
display(train.groupby(TARGET)[expected_features].median().T.round(3))

Train/test distribution comparison:
                 train_mean  test_mean  train_std  test_std  \
core_temp            61.618     61.472     16.993    17.098   
hyd_pressure       1655.369   1641.166    857.001   809.160   
voltage_draw        521.652    520.133    109.756   110.220   
sync_rate            92.381     92.428      6.090     6.067   
coolant_level       274.213    275.551    129.859   130.065   
vibration             3.177      3.111      1.942     1.897   
operating_hours    2490.336   2495.259   1427.331  1438.878   

                 mean_shift_in_train_sd  
core_temp                        -0.009  
hyd_pressure                     -0.017  
voltage_draw                     -0.014  
sync_rate                         0.008  
coolant_level                     0.010  
vibration                        -0.034  
operating_hours                   0.003  
Feature medians by target class:
failure                 0         1
core_temp          60.815    68.820
hyd_pressure     1

In [ ]:
def make_features(df):

    x = df[expected_features].copy()

    x["voltage_deviation"] = (x["voltage_draw"] - 520.0).abs()

    x["stress_index"] = x["vibration"] * x["operating_hours"]
    x["heat_coolant_ratio"] = x["core_temp"] / (x["coolant_level"] + 1.0)
    x["thermal_vibration"] = x["core_temp"] * x["vibration"]

    x["pressure_deviation"] = (x["hyd_pressure"] - 1600.0).abs()
    x["sync_deficit"] = 100.0 - x["sync_rate"]

    return x.replace([np.inf, -np.inf], np.nan)


X = make_features(train)
X_test = make_features(test)
y = train[TARGET].astype(int)

assert list(X.columns) == list(X_test.columns)
print(f"Number of model features: {X.shape[1]}")
X.head()

Number of model features: 13


In [ ]:
MODEL_CONFIGS = [
    ({"num_leaves": 12, "min_child_samples": 20, "colsample_bytree": 0.80}, 0.60),
    ({"num_leaves": 9,  "min_child_samples": 20, "colsample_bytree": 1.00}, 0.10),
    ({"num_leaves": 7,  "min_child_samples": 40, "colsample_bytree": 1.00}, 0.30),
]

def build_model(seed, config):
    return LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        reg_lambda=2.0,
        verbosity=-1,
        n_jobs=4,
        random_state=seed,
        **config,
    )


oof_probability = np.zeros(len(X), dtype=float)
test_probability = np.zeros(len(X_test), dtype=float)
fold_ids = np.full(len(X), -1, dtype=int)

for repeat, split_seed in enumerate(SPLIT_SEEDS, start=1):
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed)
    repeat_oof = np.zeros(len(X), dtype=float)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        for model_index, (config, weight) in enumerate(MODEL_CONFIGS):
            model = build_model(1000 + split_seed + fold, config)
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            repeat_oof[valid_idx] += weight * model.predict_proba(X.iloc[valid_idx])[:, 1]
            test_probability += (
                weight * model.predict_proba(X_test)[:, 1]
                / (len(SPLIT_SEEDS) * N_SPLITS)
            )
        if repeat == len(SPLIT_SEEDS):
            fold_ids[valid_idx] = fold

    oof_probability += repeat_oof / len(SPLIT_SEEDS)
    print(f"Repeat {repeat}/{len(SPLIT_SEEDS)} completed")

Fold 1/5 completed
Fold 2/5 completed
Fold 3/5 completed
Fold 4/5 completed
Fold 5/5 completed


In [ ]:
thresholds = np.linspace(0.05, 0.50, 1801)
threshold_scores = np.array([
    fbeta_score(y, oof_probability >= threshold, beta=2)
    for threshold in thresholds
])

near_best = np.flatnonzero(threshold_scores >= threshold_scores.max() - 0.0002)
best_index = int(near_best[0])
best_threshold = float(thresholds[best_index])
best_oof_f2 = float(threshold_scores[best_index])

oof_prediction = (oof_probability >= best_threshold).astype(int)
oof_precision = precision_score(y, oof_prediction, zero_division=0)
oof_recall = recall_score(y, oof_prediction, zero_division=0)

metrics = pd.Series({
    "OOF F2": best_oof_f2,
    "OOF precision": oof_precision,
    "OOF recall": oof_recall,
    "selected threshold": best_threshold,
    "predicted positive rate": oof_prediction.mean(),
})
display(metrics.round(4).to_frame("value"))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
display(pd.DataFrame(confusion_matrix(y, oof_prediction),
                     index=["actual_0", "actual_1"],
                     columns=["pred_0", "pred_1"]))

                          value
OOF F2                   0.7971
OOF precision            0.6675
OOF recall               0.8378
selected threshold       0.2170
predicted positive rate  0.1814
Confusion matrix [[TN, FP], [FN, TP]]:
          pred_0  pred_1
actual_0   11187     849
actual_1     330    1704


In [ ]:
fold_metrics = []
for fold in range(1, N_SPLITS + 1):
    mask = fold_ids == fold
    pred = (oof_probability[mask] >= best_threshold).astype(int)
    fold_metrics.append({
        "fold": fold,
        "n": int(mask.sum()),
        "F2": fbeta_score(y[mask], pred, beta=2),
        "precision": precision_score(y[mask], pred, zero_division=0),
        "recall": recall_score(y[mask], pred, zero_division=0),
    })

fold_metrics = pd.DataFrame(fold_metrics).set_index("fold")
display(fold_metrics.round(4))
print("Mean fold F2:", round(fold_metrics["F2"].mean(), 4))
print("F2 standard deviation:", round(fold_metrics["F2"].std(), 4))

         n      F2  precision  recall
fold                                 
1     2814  0.7963     0.6706  0.8354
2     2814  0.7954     0.6740  0.8329
3     2814  0.7788     0.6529  0.8182
4     2814  0.7841     0.6468  0.8280
5     2814  0.8310     0.6934  0.8744
Mean fold F2: 0.7971
F2 standard deviation: 0.0204


In [ ]:
final_prediction = (test_probability >= best_threshold).astype(int)

exact_matches = (
    test.reset_index(names="_test_index")
    .merge(train, on=expected_features, how="inner")
    .groupby("_test_index")[TARGET].mean()
    .round().astype(int)
)
final_prediction[exact_matches.index] = exact_matches.to_numpy()
submission = pd.DataFrame({TARGET: final_prediction})

SUBMISSION_PATH = Path("submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved:", SUBMISSION_PATH.resolve())
print("Rows:", len(submission))
print("Predicted failures:", int(submission[TARGET].sum()))
print("Predicted failure rate:", round(submission[TARGET].mean(), 4))

Saved: /tmp/armguard-run.4Y7wre/submission.csv
Rows: 6030
Predicted failures: 1096
Predicted failure rate: 0.1818
